# 02 ? Ridership Analysis

## ChronoTunnel: Understanding NYC Subway Demand

This notebook investigates temporal, station-level, borough-level, and hourly ridership patterns using the processed MTA datasets and ChronoTunnel production analysis functions.

**Coverage:** January 2020 ? December 2024

### Analytical objectives
- Examine yearly and monthly ridership trends.
- Evaluate year-over-year changes.
- Investigate monthly seasonality.
- Compare borough and transit-mode demand.
- Identify the highest-ridership station complexes.
- Compare sampled Wednesday and Saturday hourly patterns.
- Examine peak-period demand concentration.

**Methodology:** Reusable calculations are imported from `src/chronotunnel/analysis/`. This notebook does not redefine production analytical logic.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"

HOURLY_PATH = DATA_DIR / "hourly_ridership_sample_station_level_2020_2024.csv"
MONTHLY_PATH = DATA_DIR / "monthly_ridership_station_level_2020_2024.csv"

for path in [HOURLY_PATH, MONTHLY_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

print("Project root:", PROJECT_ROOT)

Project root: D:\ChronoTunnel


## 1. Load datasets and initialize DuckDB

The monthly dataset provides complete monthly coverage. The hourly dataset contains selected sampled dates and is used for within-day comparisons.

The two datasets are registered as DuckDB views so the existing production functions can query them.

In [2]:
hourly = pd.read_csv(HOURLY_PATH, parse_dates=["transit_timestamp"])

monthly = pd.read_csv(MONTHLY_PATH, parse_dates=["month"])

connection = duckdb.connect(database=":memory:")

connection.register("hourly_df", hourly)
connection.register("monthly_df", monthly)

connection.execute("CREATE VIEW hourly AS SELECT * FROM hourly_df")
connection.execute("CREATE VIEW monthly AS SELECT * FROM monthly_df")

print(f"Hourly records: {len(hourly):,}")
print(f"Monthly records: {len(monthly):,}")
print(f"Hourly sampled dates: {hourly['transit_timestamp'].dt.date.nunique():,}")
print(f"Monthly periods: {monthly['month'].nunique():,}")

display(monthly.head())

Hourly records: 1,172,916
Monthly records: 25,679


Hourly sampled dates: 120
Monthly periods: 60


,month,transit_mode,station_complex_id,station_complex,borough,latitude,longitude,ridership,transfers,source_rows
0,2020-01-01,subway,1,"Astoria-Ditmars Blvd (N,W)",Queens,40.775036,-73.912030,438897,13836,1
1,2020-01-01,subway,10,"49 St (N,R,W)",Manhattan,40.759900,-73.984140,556570,1414,1
2,2020-01-01,subway,100,"Hewes St (M,J)",Brooklyn,40.706870,-73.953430,94166,4807,1
3,2020-01-01,subway,101,"Marcy Av (M,J,Z)",Brooklyn,40.708360,-73.957756,396635,23843,1
4,2020-01-01,subway,103,"Bowery (J,Z)",Manhattan,40.720280,-73.993910,106995,289,1


## 2. Import production analytical functions

The following functions are maintained in the ChronoTunnel source package. Using them keeps notebook calculations consistent with the application pipeline.

In [ ]:
import sys
from pathlib import Path

# Locate the ChronoTunnel project root.
PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").exists()
        and (path / "src" / "chronotunnel").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the ChronoTunnel project root.")

# Add the project root to Python's import path.
project_root_str = str(PROJECT_ROOT)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

from src.chronotunnel.analysis.eda import (
    get_borough_ridership,
    get_monthly_ridership_trend,
    get_top_stations,
    get_transit_mode_ridership,
    get_yearly_ridership,
)
from src.chronotunnel.analysis.temporal import (
    get_monthly_seasonality,
    get_peak_hour_concentration,
    get_weekday_weekend_comparison,
    get_year_over_year_growth,
)

print("Project root:", PROJECT_ROOT)
print("Production analytical functions imported successfully.")

Project root: D:\ChronoTunnel
Production analytical functions imported successfully.


## 3. Yearly ridership trends

Aggregate ridership and transfers by calendar year using the complete monthly dataset.

In [4]:
yearly = get_yearly_ridership(connection)

display(yearly)

fig = px.bar(
    yearly,
    x="year",
    y="total_ridership",
    title="Annual NYC Subway Ridership (2020?2024)",
    labels={"year": "Year", "total_ridership": "Total Ridership"},
    text_auto=".3s",
)

fig.update_layout(template="plotly_white")
fig.show()

yearly["ridership_change_percent"] = (
    yearly["total_ridership"].pct_change() * 100
).round(2)

display(yearly)

,year,total_ridership,total_transfers
0,2020,6.433108e+08,21646280.0
1,2021,7.639145e+08,34936651.0
2,2022,1.020580e+09,42638743.0
3,2023,1.163347e+09,49398406.0
4,2024,1.211513e+09,49281662.0


,year,total_ridership,total_transfers,ridership_change_percent
0,2020,6.433108e+08,21646280.0,NaN
1,2021,7.639145e+08,34936651.0,18.75
2,2022,1.020580e+09,42638743.0,33.60
3,2023,1.163347e+09,49398406.0,13.99
4,2024,1.211513e+09,49281662.0,4.14


## 4. Monthly ridership trends

Explore the full monthly system-wide ridership series. The monthly dataset is not restricted to the 120 sampled dates used in hourly analysis.

In [5]:
monthly_trend = get_monthly_ridership_trend(connection)

monthly_trend["month"] = pd.to_datetime(monthly_trend["month"])
display(monthly_trend.head())
display(monthly_trend.tail())

fig = px.line(
    monthly_trend,
    x="month",
    y="total_ridership",
    title="Monthly NYC Subway Ridership (2020?2024)",
    labels={"month": "Month", "total_ridership": "Total Ridership"},
    markers=True,
)

fig.update_layout(template="plotly_white")
fig.show()

,month,total_ridership,total_transfers
0,2020-01-01,139044750.0,4796083.0
1,2020-02-01,131491055.0,4412832.0
2,2020-03-01,74344483.0,2505904.0
3,2020-04-01,12067953.0,78404.0
4,2020-05-01,15517576.0,87693.0


,month,total_ridership,total_transfers
55,2024-08-01,94280579.0,3818218.0
56,2024-09-01,104963114.0,3893674.0
57,2024-10-01,115158614.0,4589820.0
58,2024-11-01,104004591.0,4252916.0
59,2024-12-01,105364952.0,4263094.0


## 5. Year-over-year growth

Year-over-year growth compares each month's ridership with the same month one year earlier.

The first 12 months have no previous-year comparison and therefore have null growth values.

In [6]:
yoy = get_year_over_year_growth(connection)

display(yoy.head(15))
display(yoy.tail(15))

fig = px.line(
    yoy,
    x="month",
    y="yoy_growth_percent",
    title="Monthly Year-over-Year Ridership Change",
    labels={"month": "Month", "yoy_growth_percent": "YoY Change (%)"},
    markers=True,
)

fig.add_hline(y=0, line_dash="dash")
fig.update_layout(template="plotly_white")
fig.show()

print("Months with available YoY comparisons:", yoy["yoy_growth_percent"].notna().sum())

,month,total_ridership,previous_year_ridership,yoy_growth_percent
0,2020-01-01,139044750.0,NaN,NaN
1,2020-02-01,131491055.0,NaN,NaN
2,2020-03-01,74344483.0,NaN,NaN
3,2020-04-01,12067953.0,NaN,NaN
4,2020-05-01,15517576.0,NaN,NaN
5,2020-06-01,24031484.0,NaN,NaN
6,2020-07-01,32578207.0,NaN,NaN
7,2020-08-01,35252256.0,NaN,NaN
8,2020-09-01,42891561.0,NaN,NaN
9,2020-10-01,48249528.0,NaN,NaN


,month,total_ridership,previous_year_ridership,yoy_growth_percent
45,2023-10-01,105754112.0,95851419.0,10.33
46,2023-11-01,99812953.0,91798542.0,8.73
47,2023-12-01,98438189.0,92698422.0,6.19
48,2024-01-01,92887763.0,89746136.0,3.50
49,2024-02-01,91714598.0,85196341.0,7.65
50,2024-03-01,101543601.0,102013191.0,-0.46
51,2024-04-01,100757226.0,95006149.0,6.05
52,2024-05-01,107408577.0,105339951.0,1.96
53,2024-06-01,97874250.0,98474040.0,-0.61
54,2024-07-01,95555038.0,91966665.0,3.90


Months with available YoY comparisons: 48


## 6. Monthly seasonality

Compare average ridership for each calendar month across:
- 2020?2024.
- 2021?2024, excluding the first pandemic year.

This helps distinguish recurring seasonal patterns from the unusual conditions in 2020.

In [7]:
seasonality = get_monthly_seasonality(connection)

display(seasonality)

fig = px.line(
    seasonality,
    x="month_name",
    y=[
        "average_ridership_2020_2024",
        "average_ridership_2021_2024",
    ],
    title="Average Ridership by Calendar Month",
    labels={
        "month_name": "Calendar Month",
        "value": "Average Ridership",
        "variable": "Period",
    },
    markers=True,
)

fig.update_layout(template="plotly_white")
fig.show()

,month_number,month_name,average_ridership_2020_2024,average_ridership_2021_2024,minimum_ridership,maximum_ridership,years_observed,years_excluding_2020
0,1,January,85429924.4,7.202622e+07,41441078.0,139044750.0,5,4
1,2,February,83644797.4,7.168323e+07,39467469.0,131491055.0,5,4
2,3,March,83404506.8,8.566951e+07,51125968.0,102013191.0,5,4
3,4,April,69084453.4,8.333858e+07,12067953.0,100757226.0,5,4
4,5,May,75087325.8,8.997976e+07,15517576.0,107408577.0,5,4
5,6,June,74959705.2,8.769176e+07,24031484.0,98474040.0,5,4
6,7,July,73702175.6,8.398317e+07,32578207.0,95555038.0,5,4
7,8,August,75061549.6,8.501387e+07,35252256.0,95118263.0,5,4
8,9,September,82047127.2,9.183602e+07,42891561.0,104963114.0,5,4
9,10,October,89775346.4,1.001568e+08,48249528.0,115158614.0,5,4


## 7. Borough-level ridership

Compare total ridership, transfers, and the number of station complexes across boroughs.

In [8]:
borough = get_borough_ridership(connection)

borough["ridership_share_percent"] = (
    borough["total_ridership"] / borough["total_ridership"].sum() * 100
).round(2)

display(borough)

fig = px.bar(
    borough.sort_values("total_ridership", ascending=True),
    x="total_ridership",
    y="borough",
    orientation="h",
    title="Total Ridership by Borough (2020?2024)",
    labels={"total_ridership": "Total Ridership", "borough": "Borough"},
    text_auto=".3s",
)

fig.update_layout(template="plotly_white")
fig.show()

,borough,total_ridership,total_transfers,unique_stations,ridership_share_percent
0,Manhattan,2.576697e+09,55359465.0,123,53.65
1,Brooklyn,1.098173e+09,46569921.0,156,22.87
2,Queens,7.347860e+08,69482941.0,79,15.30
3,Bronx,3.841628e+08,24310041.0,68,8.00
4,Staten Island,8.846915e+06,2179374.0,2,0.18


## 8. Transit-mode comparison

Inspect ridership across the available transit modes. Station counts in this summary are based on distinct station identifiers within each mode.

In [9]:
transit_modes = get_transit_mode_ridership(connection)

display(transit_modes)

fig = px.bar(
    transit_modes,
    x="transit_mode",
    y="total_ridership",
    title="Ridership by Transit Mode",
    labels={"transit_mode": "Transit Mode", "total_ridership": "Total Ridership"},
    text_auto=".3s",
)

fig.update_layout(template="plotly_white")
fig.show()

,transit_mode,total_ridership,total_transfers,unique_stations
0,subway,4.783143e+09,194274478.0,425
1,tram,1.067486e+07,1447890.0,2
2,staten_island_railway,8.846913e+06,2179374.0,2


## 9. Highest-ridership station complexes

Identify the top 15 station complexes by cumulative ridership over the complete monthly dataset.

In [10]:
top_stations = get_top_stations(connection, limit=15)

display(top_stations)

fig = px.bar(
    top_stations.sort_values("total_ridership", ascending=True),
    x="total_ridership",
    y="station_complex",
    color="borough",
    orientation="h",
    title="Top 15 Station Complexes by Ridership",
    labels={
        "total_ridership": "Total Ridership",
        "station_complex": "Station Complex",
        "borough": "Borough",
    },
    text_auto=".3s",
)

fig.update_layout(
    template="plotly_white",
    yaxis={"categoryorder": "total ascending"},
)
fig.show()

,station_complex_id,station_complex,borough,total_ridership,total_transfers
0,611,"Times Sq-42 St (N,Q,R,W,S,1,2,3,7)/42 St (A,C,E)",Manhattan,168202157.0,1457527.0
1,610,"Grand Central-42 St (S,4,5,6,7)",Manhattan,115353629.0,976386.0
2,607,"34 St-Herald Sq (B,D,F,M,N,Q,R,W)",Manhattan,97401047.0,1120934.0
3,602,"14 St-Union Sq (L,N,Q,R,W,4,5,6)",Manhattan,86561174.0,1823930.0
4,628,"Fulton St (A,C,J,Z,2,3,4,5)",Manhattan,70686291.0,228000.0
5,164,"34 St-Penn Station (A,C,E)",Manhattan,68310948.0,464326.0
6,614,"59 St-Columbus Circle (A,B,C,D,1)",Manhattan,63617661.0,1135038.0
7,318,"34 St-Penn Station (1,2,3)",Manhattan,62794521.0,725122.0
8,616,"74-Broadway (7)/Jackson Hts-Roosevelt Av (E,F,...",Queens,58200078.0,5689630.0
9,447,Flushing-Main St (7),Queens,57024065.0,14453256.0


## 10. Sampled Wednesday and Saturday hourly patterns

The hourly dataset contains one Wednesday and one Saturday per month. This analysis compares average hourly system-wide ridership for those sampled days.

It does not represent every Wednesday or Saturday in the five-year period.

In [11]:
hourly_comparison = get_weekday_weekend_comparison(connection)

display(hourly_comparison.head())
display(hourly_comparison.groupby("day_name")["sampled_dates"].max())

fig = px.line(
    hourly_comparison,
    x="hour",
    y="average_ridership",
    color="day_name",
    title="Average Hourly Ridership: Sampled Wednesdays vs Saturdays",
    labels={
        "hour": "Hour of Day",
        "average_ridership": "Average Ridership",
        "day_name": "Sampled Day",
    },
    markers=True,
)

fig.update_layout(
    template="plotly_white",
    xaxis={"dtick": 2},
)
fig.show()

,day_of_week,day_name,hour,average_ridership,sampled_dates
0,3,Wednesday,0,18896.73,60
1,3,Wednesday,1,6530.50,60
2,3,Wednesday,2,3913.73,60
3,3,Wednesday,3,4588.07,60
4,3,Wednesday,4,15232.97,60


day_name
Saturday     60
Wednesday    60
Name: sampled_dates, dtype: int64

## 11. Peak-hour concentration

ChronoTunnel defines analytical peak windows as:
- Morning: 07:00?09:00.
- Evening: 16:00?18:00.

These are project-defined analysis windows, not official MTA peak-period classifications.

In [12]:
peak_concentration = get_peak_hour_concentration(connection)

display(peak_concentration)

fig = px.bar(
    peak_concentration,
    x="day_name",
    y="ridership_weighted_peak_share_percent",
    title="Ridership-Weighted Peak-Period Share",
    labels={
        "day_name": "Sampled Day",
        "ridership_weighted_peak_share_percent": "Peak Share (%)",
    },
    text_auto=".2f",
)

fig.update_layout(template="plotly_white")
fig.show()

,day_of_week,day_name,sampled_dates,average_daily_ridership,average_peak_period_ridership,ridership_weighted_peak_share_percent,average_daily_peak_share_percent
0,3,Wednesday,60,3208250.73,1579610.35,49.24,48.31
1,6,Saturday,60,2003929.13,673256.30,33.60,33.83


## 12. Analytical findings

Use the computed tables and visualizations to document:

1. How annual ridership changed from 2020 through 2024.
2. Which periods experienced the largest year-over-year changes.
3. Whether seasonal patterns are consistent across years.
4. Which boroughs account for the greatest ridership share.
5. Which station complexes have the highest cumulative demand.
6. How sampled Wednesday and Saturday hourly profiles differ.
7. What proportion of sampled daily ridership falls within the defined peak windows.

### Interpretation limitations

- Monthly analysis covers January 2020?December 2024.
- Hourly analysis uses 120 selected dates, not continuous observations.
- Wednesday/Saturday comparisons are limited to the selected sampled dates.
- Peak windows are analytical definitions chosen for this project.
- Aggregated ridership does not establish the causes of demand changes.
- Transfers are separate from ridership and should not be added to it.

### Next notebook

Continue to **03 ? Statistical Analysis** to examine distributions, statistical tests, confidence intervals, and effect sizes.